In [5]:
import os
os.chdir(r'C:\Users\progw\dashboard_energie_europe')
print("Dossier de travail :", os.getcwd())

Dossier de travail : C:\Users\progw\dashboard_energie_europe


In [10]:
import duckdb

conn = duckdb.connect('data/energy_europe.duckdb')

# Vue 1 : classement des pays par % renouvelables chaque année
conn.execute("""
    CREATE OR REPLACE VIEW v_classement_pays AS
    SELECT
        f.country,
        c.region,
        f.year,
        f.pct_renouvelable,
        f.pct_nucleaire,
        f.carbon_intensity_elec,
        RANK() OVER (
            PARTITION BY f.year
            ORDER BY f.pct_renouvelable DESC
        ) AS rang_renouvelable,
        RANK() OVER (
            PARTITION BY f.year
            ORDER BY f.carbon_intensity_elec ASC
        ) AS rang_carbone
    FROM fact_energy f
    JOIN dim_country c ON f.country = c.country_name
""")

# Vue 2 : variation annuelle par pays
conn.execute("""
    CREATE OR REPLACE VIEW v_evolution AS
    SELECT
        country,
        year,
        pct_renouvelable,
        pct_nucleaire,
        carbon_intensity_elec,
        LAG(pct_renouvelable) OVER (
            PARTITION BY country ORDER BY year
        ) AS pct_renouvelable_n1,
        ROUND(pct_renouvelable - LAG(pct_renouvelable) OVER (
            PARTITION BY country ORDER BY year
        ), 2) AS variation_renouvelable,
        LAG(carbon_intensity_elec) OVER (
            PARTITION BY country ORDER BY year
        ) AS carbone_n1,
        ROUND(carbon_intensity_elec - LAG(carbon_intensity_elec) OVER (
            PARTITION BY country ORDER BY year
        ), 1) AS variation_carbone
    FROM fact_energy
""")

# Vue 3 : résumé par région et décennie
conn.execute("""
    CREATE OR REPLACE VIEW v_resume_region AS
    SELECT
        c.region,
        d.decade,
        ROUND(AVG(f.pct_renouvelable), 1)      AS pct_renouvelable_moyen,
        ROUND(AVG(f.pct_nucleaire), 1)          AS pct_nucleaire_moyen,
        ROUND(AVG(f.carbon_intensity_elec), 0) AS intensite_carbone_moyenne,
        COUNT(DISTINCT f.country)              AS nb_pays
    FROM fact_energy f
    JOIN dim_country c ON f.country = c.country_name
    JOIN dim_date    d ON f.year    = d.year_id
    GROUP BY c.region, d.decade
    ORDER BY c.region, d.decade
""")

print("Vues créées :")
print(conn.execute("SHOW TABLES").fetchdf())

print("\nTop 5 pays les plus renouvelables en 2023 :")
print(conn.execute("""
    SELECT country, region, pct_renouvelable, pct_nucleaire, rang_renouvelable
    FROM v_classement_pays
    WHERE year = 2023
    ORDER BY rang_renouvelable
    LIMIT 5
""").fetchdf())

print("\nTop 5 pays les plus nucléaires en 2023 :")
print(conn.execute("""
    SELECT country, region, pct_nucleaire, pct_renouvelable, rang_carbone
    FROM v_classement_pays
    WHERE year = 2023
    ORDER BY pct_nucleaire DESC
    LIMIT 5
""").fetchdf())

print("\nRésumé par région :")
print(conn.execute("SELECT * FROM v_resume_region").fetchdf())

Vues créées :
                name
0        dim_country
1           dim_date
2        fact_energy
3  v_classement_pays
4        v_evolution
5    v_resume_region

Top 5 pays les plus renouvelables en 2023 :
    country region  pct_renouvelable  pct_nucleaire  rang_renouvelable
0    Norway   Nord              98.5            0.0                  1
1   Denmark   Nord              86.4            0.0                  2
2   Austria  Ouest              84.8            0.0                  3
3  Portugal    Sud              74.5            0.0                  4
4   Croatia    Est              70.1            0.0                  5

Top 5 pays les plus nucléaires en 2023 :
    country region  pct_nucleaire  pct_renouvelable  rang_carbone
0    France  Ouest           65.2              26.9             4
1   Hungary    Est           44.8              26.4            12
2   Finland   Nord           42.1              51.9             5
3  Bulgaria    Est           40.3              25.1           

In [13]:
import pandas as pd
import os

# Table dimension pays
dim_country = pd.DataFrame({
    'country': [
        'France', 'Germany', 'Spain', 'Italy', 'Poland',
        'Sweden', 'Norway', 'Netherlands', 'Belgium', 'Austria',
        'Portugal', 'Denmark', 'Finland', 'Czechia', 'Romania',
        'Hungary', 'Switzerland', 'Greece', 'Bulgaria', 'Croatia',
        'Ireland', 'United Kingdom'
    ],
    'region': [
        'Ouest', 'Ouest', 'Sud', 'Sud', 'Est',
        'Nord', 'Nord', 'Ouest', 'Ouest', 'Ouest',
        'Sud', 'Nord', 'Nord', 'Est', 'Est',
        'Est', 'Ouest', 'Sud', 'Est', 'Est',
        'Nord', 'Autre'
    ]
})

# Table dimension année
dim_year = pd.DataFrame({
    'year': range(2000, 2026),
    'decade': ['2000s']*10 + ['2010s']*10 + ['2020s']*6,
    'period': ['avant 2010']*10 + ['2010-2019']*10 + ['2020+']*6
})

# Export
dim_country.to_csv('exports/dim_country.csv', index=False)
dim_year.to_csv('exports/dim_year.csv', index=False)

print("Tables de dimension exportées :")
print(f"  dim_country : {len(dim_country)} pays")
print(f"  dim_year : {len(dim_year)} années")

Tables de dimension exportées :
  dim_country : 22 pays
  dim_year : 26 années


In [15]:
# Création du dossier exports si nécessaire
os.makedirs('exports', exist_ok=True)

# Export des 4 fichiers CSV pour Power BI
conn.execute("COPY v_classement_pays TO 'exports/export_classement.csv' (HEADER, DELIMITER ',')")
conn.execute("COPY v_evolution       TO 'exports/export_evolution.csv'  (HEADER, DELIMITER ',')")
conn.execute("COPY v_resume_region   TO 'exports/export_region.csv'     (HEADER, DELIMITER ',')")
conn.execute("COPY fact_energy       TO 'exports/export_faits.csv'      (HEADER, DELIMITER ',')")

# Vérification
print("Fichiers exportés :")
for f in os.listdir('exports'):
    taille = os.path.getsize(f'exports/{f}')
    print(f"  {f} — {taille} octets")

# Fermeture propre de la connexion
conn.close()
print("\nConnexion DuckDB fermée — notebooks terminés !")

ConnectionException: Connection Error: Connection already closed!

In [16]:
import pandas as pd

# Table dimension pays
dim_country = pd.DataFrame({
    'country': [
        'France', 'Germany', 'Spain', 'Italy', 'Poland',
        'Sweden', 'Norway', 'Netherlands', 'Belgium', 'Austria',
        'Portugal', 'Denmark', 'Finland', 'Czechia', 'Romania',
        'Hungary', 'Switzerland', 'Greece', 'Bulgaria', 'Croatia',
        'Ireland', 'United Kingdom'
    ],
    'region': [
        'Ouest', 'Ouest', 'Sud', 'Sud', 'Est',
        'Nord', 'Nord', 'Ouest', 'Ouest', 'Ouest',
        'Sud', 'Nord', 'Nord', 'Est', 'Est',
        'Est', 'Ouest', 'Sud', 'Est', 'Est',
        'Nord', 'Autre'
    ]
})

# Table dimension année
dim_year = pd.DataFrame({
    'year': range(2000, 2026),
    'decade': ['2000s']*10 + ['2010s']*10 + ['2020s']*6,
    'period': ['avant 2010']*10 + ['2010-2019']*10 + ['2020+']*6
})

# Export direct sans DuckDB
dim_country.to_csv('exports/dim_country.csv', index=False)
dim_year.to_csv('exports/dim_year.csv', index=False)

print("Tables de dimension exportées :")
print(f"  dim_country : {len(dim_country)} pays")
print(f"  dim_year : {len(dim_year)} années")

Tables de dimension exportées :
  dim_country : 22 pays
  dim_year : 26 années
